# 📄 PDF Q&A Pipeline using RAG

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that:
1. Reads and extracts text from a PDF
2. Splits it into overlapping chunks
3. Converts chunks into vector embeddings
4. Stores them in a ChromaDB vector database
5. Answers natural language questions using LLaMA 2 (via Ollama)

**Tech Stack:** Python · pdfminer · LangChain · HuggingFace · ChromaDB · Ollama (LLaMA 2)

## 1. Install Dependencies

Run this cell once to install all required libraries.

In [ ]:
!pip install pdfminer.six langchain langchain-community langchain-huggingface sentence-transformers chromadb --quiet

## 2. Import Libraries

In [ ]:
from pdfminer.high_level import extract_text
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA

print('✅ All libraries imported successfully.')

## 3. Load and Extract Text from PDF

`pdfminer` extracts all text from the PDF as a single string.  
Update `PDF_PATH` to point to your own PDF file.

In [ ]:
PDF_PATH = "/Users/shruthiraghavan/Desktop/CS 131 notes/04-25su-cs131-permissions_processes.pdf"

raw_text = extract_text(PDF_PATH)

print(f'✅ Extracted {len(raw_text)} characters from PDF.')
print('\n--- Preview (first 500 characters) ---')
print(raw_text[:500])

## 4. Convert to LangChain Document

LangChain tools expect a `Document` object rather than a plain string.  
We wrap the extracted text into one `Document` and store it in a list.

In [ ]:
documents = [Document(page_content=raw_text)]

print(f'✅ Created {len(documents)} LangChain Document(s).')

## 5. Split Text into Chunks

`RecursiveCharacterTextSplitter` splits on paragraphs → lines → sentences → words in order, so it never cuts awkwardly mid-sentence.

- `chunk_size=500` — max 500 characters per chunk  
- `chunk_overlap=50` — consecutive chunks share 50 characters so context isn't lost at edges

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f'✅ Split into {len(chunks)} chunks.')
print('\n--- Sample Chunk ---')
print(chunks[0].page_content)

## 6. Generate Embeddings and Store in ChromaDB

Each chunk is converted into a vector (list of numbers) using a HuggingFace sentence-transformer model.  
These vectors are stored in ChromaDB so we can later search for the chunks most relevant to a question.

- Model: `all-MiniLM-L6-v2` — lightweight, fast, and accurate for semantic search  
- No `persist_directory` — stored in memory only (cleared when kernel restarts)

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_documents(chunks, embedding)

print(f'✅ Stored {len(chunks)} chunks in ChromaDB vector store.')

## 7. Set Up LLM and Build QA Chain

We use **LLaMA 2 running locally via Ollama** — no API key or internet connection needed.

> ⚠️ Make sure Ollama is running before this cell:  
> `ollama pull llama2` then `ollama serve` in a terminal.

`RetrievalQA` ties everything together:  
question → retrieve top 4 relevant chunks → pass to LLM → return answer

In [ ]:
llm = Ollama(model="llama2")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=False
)

print('✅ QA chain ready.')

## 8. Ask Questions

Change the `question` variable to ask anything about your PDF.

In [ ]:
question = "What is chmod used for?"

result = qa_chain.invoke({"query": question})

print(f'Question: {question}')
print(f'\nAnswer: {result["result"]}')

## 9. Try More Questions

Ask as many questions as you like — the vector store stays in memory until the kernel restarts.

In [ ]:
questions = [
    "What are file permissions in Linux?",
    "What is the difference between a process and a thread?",
    "How does the chmod command work?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print(f'Q: {q}')
    print(f'A: {result["result"]}')
    print('-' * 60)